In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from PIL import Image
import random
import time

# =============================================================================
# ✨ OSCD RGB 데이터셋을 활용한 초급 AI 실습 스크립트 ✨
# =============================================================================
# 📚 제목: 위성사진을 이용한 '변화 감지(Change Detection)' 시각화
# 🗺️ 의미: 이 데이터셋은 '시간이 지나면서 특정 지역에 무슨 변화가 생겼는지'
#      (예: 새 건물 건축, 도로 건설)를 감지하는 AI를 훈련하는 데 사용됩니다.
# 🛰️ 사용 기술: 원격 탐사 이미지 처리, NumPy 배열 연산, 시각화 (Pre-processing 단계)
# -----------------------------------------------------------------------------
# [튜터 코멘트]: 안녕! 👋 오늘은 진짜 신나고 멋진 AI 프로젝트를 해볼 거예요.
#             우리가 만지는 데이터는 단순한 사진이 아니라, 지구 전체를 담은
#             '위성사진'이라는 거랍니다! 😲
#             우리의 목표는 과거 사진(image1)과 현재 사진(image2)를 비교해서,
#             '변한 부분(Change)'을 찾아내는 방법을 시뮬레이션하는 거예요.
#             이 과정을 통해 AI 모델이 학습하기 전에 데이터를 어떻게 다듬어야 하는지
#             배우게 될 거예요! 파이팅! 💪

# --- 설정 값 ---
DATASET_NAME = 'blanchon/OSCD_RGB'
SAMPLE_COUNT = 10  # 전체 데이터를 다 쓸 필요 없이, 딱 10개의 샘플만 볼게요!
SPLIT_NAME = 'train'

print("="*80)
print(f"🚀 [프로젝트 시작] '{DATASET_NAME}' 데이터셋 로딩을 시도합니다.")
print(f"💡 목표: {SPLIT_NAME} 스플릿에서 {SAMPLE_COUNT}개의 샘플을 가져와 변화를 분석합니다.")
print("="*80)

# 1. 데이터셋 로드 전략: 스트리밍 모드 우선 확인 (Constraint 2, 3, 18)
try:
    # 스트리밍 모드로 시도 (데이터가 클 경우 메모리 효율적)
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=True)
    print("✅ 스트리밍 모드 (Streaming)로 데이터셋 로드를 성공했습니다. (가장 빠르고 메모리 친화적!)")

except Exception as e:
    print(f"⚠️ 스트리밍 로드 중 오류 발생 ({e}). 일반 모드로 전환합니다. (일부 샘플만 다운로드하여 진행)")
    # 스트리밍 실패 시, 일반 모드(streaming=False)로 소량 다운로드하여 안전하게 진행
    try:
        dataset = load_dataset(DATASET_NAME, split=SPLIT_NAME, streaming=False)
        print("✅ 일반 모드 (Non-Streaming)로 데이터셋 로드에 성공했습니다.")
    except Exception as e2:
        print(f"🚨 데이터셋 로드 실패: {e2}. 코드를 실행할 수 없습니다.")
        exit()

# 2. 데이터셋 샘플링을 위한 반복자(Iterator) 설정 (Constraint 9, 16)
print("\n⏳ {SAMPLE_COUNT}개 샘플을 가져오기 위한 반복자를 준비합니다...")
# iterable dataset을 처리하기 위해 .take() 메소드 사용
sample_iterator = dataset.take(SAMPLE_COUNT)

# 3. 데이터 전처리 및 변화 감지 시뮬레이션
print("\n🧠 데이터 전처리 및 '변화 지표' 계산 시뮬레이션 시작...")

# 변수 초기화: 시각화에 사용할 첫 번째 샘플의 데이터를 저장합니다.
sample_data_list = []

# 반복자를 순회하며 샘플 데이터를 추출합니다.
for i, sample in enumerate(sample_iterator):
    sample_data_list.append(sample)

print("\n🎉 성공적으로 {}개의 샘플 데이터를 준비했습니다!".format(len(sample_data_list)))

# 첫 번째 샘플만 예시 분석에 사용합니다. (가장 대표적인 샘플 하나만 골라 시각화)
if sample_data_list:
    sample = sample_data_list[0]
    print("\n=================================================================")
    print(f"🎨 [시각화 예시] {i+1}번째 샘플의 변화를 분석합니다.")
    print("-----------------------------------------------------------------")

    try:
        # 1단계: 이미지 데이터 추출 및 Numpy 배열 변환 (Constraint 17)
        # PIL.Image 객체는 바로 계산에 쓸 수 없으므로, numpy로 변환합니다.
        # 모든 이미지는 (Height, Width, Channels) 튜플 형태가 되어야 연산이 가능합니다.
        img1 = np.array(sample['image1'])
        img2 = np.array(sample['image2'])
        mask = np.array(sample['mask'])

        print(f"✨ [형태 확인] Image 1: {img1.shape}, Image 2: {img2.shape}, Mask: {mask.shape}")

        # 2단계: 차원 맞추기 (Image 2가 흑백일 경우 3차원 배열로 확장) (Constraint 15)
        # 흑백 이미지 (H, W)를 색상 이미지와 같은 차원 (H, W, 3)으로 확장합니다.
        if len(img2.shape) == 2:
            img2_3d = np.expand_dims(img2, axis=-1).repeat(3, axis=-1)
        else:
            img2_3d = img2
            
        # 3단계: 변화 지표 계산 (Core AI Logic Simulation)
        # 두 이미지를 픽셀 단위로 빼서 '변화의 크기(Change Magnitude)'를 계산합니다.
        # 픽셀 값이 클수록 변화가 컸다는 의미입니다.
        # (주의: 실제 원격 탐사에서는 전처리(정규화, 대기 보정)가 더 많이 필요합니다!)
        change_difference = np.abs(img1.astype(np.float32) - img2_3d.astype(np.float32))
        
        # 4단계: 결과를 시각화하여 변화를 확인합니다.
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # 1. 과거 이미지 (Image 1)
        axes[0].imshow(np.clip(img1, 0, 255))
        axes[0].set_title("Image 1 (Before)")
        axes[0].axis('off')

        # 2. 현재 이미지 (Image 2)
        axes[1].imshow(np.clip(img2_3d[:, :, 0], 0, 255)) # RGB 중 첫 채널만으로 표시
        axes[1].set_title("Image 2 (After)")
        axes[1].axis('off')
        
        # 3. 변화 지표 맵 (Difference Map)
        # np.log나 np.mean 등을 사용하여 차원 축소 후 시각화할 수도 있습니다.
        axes[2].imshow(change_difference.mean(axis=2) / 255, cmap='hot')
        axes[2].set_title("Change Difference Map (Difference Magnitude)")
        axes[2].axis('off')
        
        plt.suptitle("🚀 Change Detection Simulation: Comparing Time 1 and Time 2 Imagery", fontsize=16)
        plt.show()
        
        print("\n💡 분석 결과: 빨갛거나 밝게 보이는 부분은 두 이미지 간에 픽셀 값의 변화가 크다는 뜻입니다.")
        print("🧐 Ground Truth Mask와 비교해 보세요. AI는 이 '변화 지표 맵'을 보고,")
        print("     실제 변화가 있었는지 (마스크가 1인 부분) 판단하도록 학습합니다.")

    except Exception as e:
        print(f"❌ 데이터 처리 또는 시각화 중 오류가 발생했습니다: {e}")

print("="*80)
print("✨ 오늘의 AI 코딩 실습을 마칩니다!")
print("🎉 정말 잘하셨어요! 위성 데이터는 일반 사진보다 전처리 과정이 훨씬 까다로워요. 다음에는 '왜 이 지역만 변화했을까?'라는 질문을 던지며, 데이터가 부족한 이유를 분석하는 능력을 기르는 게 진짜 전문가의 AI 실력이라는 것을 기억해주세요! 💪")
print("="*80)